# TP 1A — Faits stylisés : le diagnostic descriptif

**Séance 1, partie 5 · 50 minutes · en binômes · déposé en fin de partie**


### Objectifs

1. Tableau de statistiques descriptives comparées — et le choix d'annualisation
2. Graphiques quantile-quantile contre la normale et contre la Student
3. Instabilité de la kurtosis et indice de queue (Hill)
4. Corrélogrammes de $r_t$ et de $|r_t|$

### Rappel du cours

> Le rendement de demain est imprévisible ; **son amplitude ne l'est pas.**

### Ce qui est noté

Le notebook exécuté **et** la cellule de conclusion en fin de TP — 25 % de la
note, déposée avant la fin de la partie. Les **tests d'efficience** (ADF, KPSS,
ratio de variance) font l'objet du TP 1B, en séance 2, et seront notés à part.


---

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# retrouve outils_cours.py, que le notebook soit ouvert depuis 02-TP/ ou 02-TP/corriges/
for _cand in (Path.cwd(), *Path.cwd().parents[:3]):
    if (_cand / "outils_cours.py").exists():
        sys.path.insert(0, str(_cand))
        break
import outils_cours as oc

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": .3})
np.random.seed(2026)
print("Environnement prêt.")

from statsmodels.tsa.stattools import adfuller, kpss, acf

prix = oc.charger_panier()
r = oc.rendements_log(prix)
print(f"{len(r)} rendements, {prix.shape[1]} actifs.")

## 1. Statistiques descriptives — et le piège du 24/7

Le panier mélange des actifs à 365 jours de cotation (cryptos) et à 252 jours
(S&P 500, or). Utiliser le même facteur d'annualisation pour tous est une erreur.

In [ ]:
jours = {"Bitcoin": 365, "Ethereum": 365, "Solana": 365, "SP500": 252, "Or": 252}
table = oc.table_faits_stylises(r, jours_an=jours)
table.round(3)

**Question 1.1** — Recalculez la volatilité annualisée du Bitcoin en utilisant
252 jours au lieu de 365. De combien vous trompez-vous, en points et en pourcentage ?

In [ ]:
# À COMPLÉTER
# Comparer la volatilité annualisée du Bitcoin sous √365 et sous √252.
# Afficher : les deux valeurs, l'écart en points, la sous-estimation relative,
# et vérifier que le rapport vaut bien √(365/252).


## 2. La distribution n'est pas normale

Deux graphiques quantile-quantile : contre la normale, puis contre une Student
dont le degré de liberté est estimé sur les données.

In [ ]:
actif = "Bitcoin"          # <- changez-le pour explorer
x = r[actif].dropna()
z = (x - x.mean()) / x.std(ddof=1)

nu, _, _ = stats.t.fit(z, floc=0, fscale=1)
print(f"{actif} — degré de liberté estimé : ν = {nu:.2f}")
print(f"kurtosis en excès observée         : {stats.kurtosis(x):.2f}")
if nu > 4:
    print(f"kurtosis théorique d'une t(ν)      : {6/(nu-4):.2f}")
else:
    print("ν ≤ 4 → la kurtosis théorique de la Student est INFINIE.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
stats.probplot(z, dist="norm", plot=axes[0])
axes[0].set_title(f"{actif} — QQ contre la normale")
stats.probplot(z, dist=stats.t, sparams=(nu,), plot=axes[1])
axes[1].set_title(f"{actif} — QQ contre une Student(ν={nu:.1f})")
for a in axes:
    a.get_lines()[0].set(marker=".", markersize=3, alpha=.6)
plt.tight_layout(); plt.show()

**Question 2.1** — Sur le graphique de gauche, où les points s'écartent-ils
de la droite ? Que signifie physiquement cet écart pour un gérant de risque ?

**Question 2.2** — La Student ajuste-t-elle mieux ? Sur quelle partie de la
distribution reste-t-elle insuffisante, et pourquoi ?

*Vos réponses :*

>

## 3. La kurtosis se stabilise-t-elle ?

Si le moment d'ordre 4 existe, la kurtosis empirique converge quand l'échantillon
grandit. Si elle **continue de croître**, c'est que le moment n'existe pas.

In [ ]:
# À COMPLÉTER
# Tracer la kurtosis en excès calculée sur les n premières observations,
# pour n allant de 250 à la taille totale par pas de 100.
# Conclure : se stabilise-t-elle ?


**Question 3.1** — Estimez l'indice de queue par la méthode de Hill, pour
plusieurs valeurs de $k$. Y a-t-il une zone de stabilité ? Que vaut $\alpha$ ?
Le moment d'ordre 4 existe-t-il ?

In [ ]:
# À COMPLÉTER
# Tracer alpha_hat de Hill en fonction de k (utiliser oc.hill).
# Ajouter les lignes de repère alpha = 2 et alpha = 4.
# Conclure sur l'existence du moment d'ordre 4.


## 4. Dépendance : rendements vs amplitudes

C'est le fait stylisé qui rend possible toute la séance 2.

In [ ]:
n_lags = 40
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, serie, titre in [
    (axes[0], x, "rendements  $r_t$"),
    (axes[1], x.abs(), "amplitudes  $|r_t|$"),
]:
    vals = acf(serie, nlags=n_lags, fft=True)[1:]
    ax.bar(range(1, n_lags + 1), vals, width=.7)
    ic = 1.96 / np.sqrt(len(serie))
    ax.axhline(ic, color="crimson", ls="--", lw=.9)
    ax.axhline(-ic, color="crimson", ls="--", lw=.9)
    ax.set_title(f"{actif} — autocorrélation des {titre}")
    ax.set_xlabel("retard (jours)")
    ax.set_ylim(-0.15, max(0.4, vals.max() * 1.2))
plt.tight_layout(); plt.show()

**Question 4.1** — Combien de retards sont significatifs pour $r_t$ ?
Pour $|r_t|$ ? Formulez la conclusion en une phrase.

*Votre réponse :*

>

**Question 4.2 (bonus)** — Testez l'effet de levier : corrélation entre le
rendement du jour $t$ et la volatilité réalisée des 5 jours suivants.
Comparez Bitcoin et S&P 500. Le résultat est-il celui attendu ?

In [ ]:
# À COMPLÉTER
# Pour Bitcoin puis SP500 : corrélation entre r_t et l'écart-type des
# rendements sur [t+1, t+5]. Indice : .rolling(5).std().shift(-5)


---

## Votre conclusion

Sur votre panier, en une douzaine de lignes :

1. Quel **facteur d'annualisation** avez-vous retenu, et pourquoi ?
2. La kurtosis est-elle **stable** quand l'échantillon change ? Qu'en déduisez-vous
   sur le moment d'ordre 4, et sur les mesures de risque à privilégier ?
3. Que montrent les deux corrélogrammes — celui de $r_t$ et celui de
   $\lvert r_t\rvert$ ? En une phrase de gestionnaire.
4. Une différence nette entre crypto et actif traditionnel : laquelle, et de
   quelle ampleur ?

> « La kurtosis est de 14 » est une lecture, pas une interprétation.

> **C'est cette cellule qui fait l'essentiel de la note.** On attend un jugement
> argumenté, pas une liste de résultats. Écrivez ci-dessous, en markdown.

*(votre réponse ici — remplacez cette ligne)*


---

## Dépôt — Diagnostic descriptif · 25 %

**Il vous reste 5 minutes.** Dans l'ordre :

1. `Kernel` → **Restart & Run All** — un notebook qui ne s'exécute pas n'est pas corrigé
2. `File` → `Save and Export Notebook As…` → **HTML**
3. Déposer sur l'ENT : `NOM1-NOM2_diagnostic-descriptif.html`

::: Vous n'avez pas fini ?
Déposez en l'état. Un notebook incomplet, exécutable et honnêtement conclu vaut
mieux qu'un notebook complet et faux.
:::


---

::: Gardez ce notebook
Le TP 1B, en séance 2, repart du même panier. Notez vos chiffres clés :
kurtosis en excès, zone stable du graphique de Hill, autocorrélation de
$\lvert r\rvert$ au retard 1.
:::
